In [26]:
import os
import pandas as pd
from bs4 import BeautifulSoup
import lxml
from io import StringIO

In [5]:
SCORES_DIR = "data2/scores"
boxscores = os.listdir(SCORES_DIR)

In [12]:
len(boxscores)
boxscores

['.ipynb_checkpoints',
 '201512010BRK.html',
 '201512010CLE.html',
 '201512010MIN.html',
 '201512010NOP.html',
 '201512010PHI.html',
 '201512010POR.html',
 '201512020ATL.html',
 '201512020CHI.html',
 '201512020CHO.html',
 '201512020DET.html',
 '201512020HOU.html',
 '201512020LAC.html',
 '201512020NYK.html',
 '201512020SAS.html',
 '201512020WAS.html',
 '201512030MEM.html',
 '201512030MIA.html',
 '201512030POR.html',
 '201512030SAC.html',
 '201512030TOR.html',
 '201512030UTA.html',
 '201512040ATL.html',
 '201512040DAL.html',
 '201512040DET.html',
 '201512040NOP.html',
 '201512040NYK.html',
 '201512040WAS.html',
 '201512050CHI.html',
 '201512050HOU.html',
 '201512050LAC.html',
 '201512050MIA.html',
 '201512050MIL.html',
 '201512050MIN.html',
 '201512050PHI.html',
 '201512050SAS.html',
 '201512050TOR.html',
 '201512050UTA.html',
 '201512060BRK.html',
 '201512060DET.html',
 '201512060MEM.html',
 '201512060OKC.html',
 '201512060WAS.html',
 '201512070CHI.html',
 '201512070CHO.html',
 '2015120

In [13]:
box_scores = [os.path.join(SCORES_DIR, f).replace("\\", "/") for f in boxscores if ".html" in f ]


['data2/scores/201512010BRK.html',
 'data2/scores/201512010CLE.html',
 'data2/scores/201512010MIN.html',
 'data2/scores/201512010NOP.html',
 'data2/scores/201512010PHI.html',
 'data2/scores/201512010POR.html',
 'data2/scores/201512020ATL.html',
 'data2/scores/201512020CHI.html',
 'data2/scores/201512020CHO.html',
 'data2/scores/201512020DET.html',
 'data2/scores/201512020HOU.html',
 'data2/scores/201512020LAC.html',
 'data2/scores/201512020NYK.html',
 'data2/scores/201512020SAS.html',
 'data2/scores/201512020WAS.html',
 'data2/scores/201512030MEM.html',
 'data2/scores/201512030MIA.html',
 'data2/scores/201512030POR.html',
 'data2/scores/201512030SAC.html',
 'data2/scores/201512030TOR.html',
 'data2/scores/201512030UTA.html',
 'data2/scores/201512040ATL.html',
 'data2/scores/201512040DAL.html',
 'data2/scores/201512040DET.html',
 'data2/scores/201512040NOP.html',
 'data2/scores/201512040NYK.html',
 'data2/scores/201512040WAS.html',
 'data2/scores/201512050CHI.html',
 'data2/scores/20151

In [153]:
soup


<h1>Cleveland Cavaliers at Atlanta Hawks Box Score, April 1, 2016</h1>
<div class="section_wrapper" id="all_other_scores">
<div class="section_heading assoc_other_scores" id="other_scores_sh">
<span class="section_anchor" data-label="All Games This Date" id="other_scores_link"></span><h2></h2>
</div><div class="placeholder"></div>
<!--     <div class="section_content" id="div_other_scores">
	    <div class="game_summaries compressed">
   <h2>NBA Scores &mdash; Apr 1, 2016</h2>
   
      <div class="game_summary nohover current ">
	<table class="teams poptip" data-tip="Cleveland Cavaliers at Atlanta Hawks">
		<tbody>       
		<tr class="winner">
			<td><a href="/teams/CLE/2016.html">CLE</a></td>
			<td class="right">110</td>
			<td class="right gamelink">
				<a href="/boxscores/201604010ATL.html">F<span class="no_mobile">inal</span></a>
				
			</td>
		</tr>
		<tr class="loser">
			<td><a href="/teams/ATL/2016.html">ATL</a></td>
			<td class="right">108</td>
			<td class="right">OT
		

In [137]:
pip install lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [138]:
teams = list(line_scores["Teams"])
teams

['PHO', 'BRK']

In [161]:
# for team in teams:

base_cols = None
games = []

def read_stats(soup,team,stat):
    stats = pd.read_html(StringIO(str(soup)),attrs = {"id":f"box-{team}-game-{stat}"},index_col=0)[0]
    stats = stats.apply(pd.to_numeric,errors="coerce")
    return stats

    
def read_season_info(soup):
    nav = soup.select("#bottom_nav_container")[0]
    hrefs = [a["href"] for a in nav.find_all("a")]
    season = os.path.basename(hrefs[1]).split("_")[0]
    return season


def read_line_score(soup):
    line_score = pd.read_html(StringIO(str(soup)),attrs ={"id":"line_score"})[0]
    cols = list(line_score.columns)
    cols[0] = "Teams"
    cols[-1] = "Total"
    line_score.columns = cols
    line_score = line_score[["Teams","Total"]]
    return line_score

def parse_html(boxscore):
    with open(boxscore,"r",encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html)
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    return soup



for boxscore in box_scores:
    soup = parse_html(boxscore)
    line_score = read_line_score(soup)
    teams = list(line_score["Teams"])
    summaries=[]
    for team in teams:
        
        basic = read_stats(soup,team,"basic")   
        advanced = read_stats(soup,team,"advanced")
        totals = pd.concat([basic.iloc[-1,:],advanced.iloc[-1,:]])
        totals.index = totals.index.str.lower()
     
        maxes = pd.concat([basic.iloc[:-1,:].max(),advanced.iloc[:-1,:].max()])
        maxes.index = maxes.index.str.lower() + "_max"
        summary = pd.concat([totals,maxes])
     
        if base_cols is None:
             base_cols = list(summary.index.drop_duplicates(keep="first"))
             base_cols = [l for l in base_cols if "bpm" not in l]
        summary = summary[base_cols]
        summaries.append(summary)
    summary = pd.concat(summaries,axis=1).T
   
    summary["home"] = [0,1]
    
    game = pd.concat([summary,line_score],axis=1)
    game_opp = game.iloc[::-1].reset_index()
    game_opp.columns = game_opp.columns + "_opp"
    full_game = pd.concat([game,game_opp],axis=1)
        
    full_game["season"] = read_season_info(soup)
    full_game["date"] = os.path.basename(boxscore)[:8]
    full_game["date"] = pd.to_datetime(full_game["date"],format="%Y%m%d")
    full_game["won"] = full_game["Total"]>full_game["Total_opp"]
    games.append(full_game)
    
    if len(games) % 100 == 0:
        print("hey bro")

    
# basic
# totals
# maxes



hey bro
hey bro
hey bro
hey bro
hey bro


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x97 in position 6447: invalid start byte

In [163]:
games_df = pd.concat(games, ignore_index=True)
len(games_df)

1048

In [165]:
games_df.to_csv("games.csv",index=False)

In [154]:
games

[      mp     mp    fg   fga    fg%    3p   3pa    3p%    ft   fta  ...  \
 0  240.0  240.0  33.0  75.0  0.440  10.0  28.0  0.357  15.0  22.0  ...   
 1  240.0  240.0  41.0  84.0  0.488   4.0  13.0  0.308   8.0  12.0  ...   
 
    tov%_max_opp  usg%_max_opp  ortg_max_opp  drtg_max_opp  home_opp  \
 0          33.3          30.2         139.0         103.0         1   
 1          33.3          30.5         125.0         105.0         0   
 
    Teams_opp  Total_opp  season       date    won  
 0        BRK         94    2016 2015-12-01  False  
 1        PHO         91    2016 2015-12-01   True  
 
 [2 rows x 154 columns],
       mp     mp    fg   fga    fg%   3p   3pa   3p%    ft   fta  ...  \
 0  240.0  240.0  39.0  83.0  0.470  7.0  28.0  0.25  12.0  16.0  ...   
 1  240.0  240.0  28.0  83.0  0.337  9.0  30.0  0.30  20.0  23.0  ...   
 
    tov%_max_opp  usg%_max_opp  ortg_max_opp  drtg_max_opp  home_opp  \
 0          57.1          37.6         153.0         104.0         1   
 1  